# PRUEBA CON S&P500

In [ ]:
!pip install yfinance pandas_ta PyPortfolioOpt PyPortfolioOpt ray pandas_datareader

In [4]:
# %%writefile RollingOLSRegression.py

from statsmodels.regression.rolling import RollingOLS
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
from statsmodels.regression.rolling import RollingOLS
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
import datetime as dt
import yfinance as yf
import pandas_ta
import warnings
import copy
from sklearn.cluster import KMeans
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns
import ray
import time

warnings.filterwarnings('ignore')

class RollingOLSRegression:

  def __init__(self,symbols_list,start_date,end_date):
    self.model=None;
    self.symbols_list=symbols_list;
    self.start_date=start_date;
    self.end_date=end_date;


  def load_data(self):
    data= yf.download(tickers=self.symbols_list,
                 start=self.start_date,
                 end=self.end_date,auto_adjust=False)
    return data

  #paralelizable
  def calculate_tecnical_indicators(self,data):
    df=copy.deepcopy(data)
    df=df.stack()
    df.index.names=['date','ticker']
    df.columns=df.columns.str.lower()


    # calculate Garman-Klass Volatility, rsi,bollinger bands, atr,macd,dollar volume
    df['garman_klass_vol'] = ((np.log(df['high'])-np.log(df['low']))**2)/2-(2*np.log(2)-1)*((np.log(df['adj close'])-np.log(df['open']))**2)

    df['rsi'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.rsi(close=x, length=20))

    df['bb_low'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,0])

    df['bb_mid'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,1])

    df['bb_high'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,2])

    def compute_atr(stock_data):
        atr = pandas_ta.atr(high=stock_data['high'],
                            low=stock_data['low'],
                            close=stock_data['close'],
                            length=14)
        return atr.sub(atr.mean()).div(atr.std())

    df['atr'] = df.groupby(level=1, group_keys=False).apply(compute_atr)

    def compute_macd(close):
        macd = pandas_ta.macd(close=close, length=20).iloc[:,0]
        return macd.sub(macd.mean()).div(macd.std())

    df['macd'] = df.groupby(level=1, group_keys=False)['adj close'].apply(compute_macd)

    df['dollar_volume'] = (df['adj close']*df['volume'])/1e6

    return df

  def aggregate_to_monthly_level(self,df):

    # To reduce training time and experiment with features and strategies,
    # we convert the business-daily data to month-end frequency.


    last_cols = [c for c in df.columns.unique(0) if c not in ['dollar_volume', 'volume', 'open',
                                                            'high', 'low', 'close']]

    data = (pd.concat([df.unstack('ticker')['dollar_volume'].resample('M').mean().stack('ticker').to_frame('dollar_volume'),
                    df.unstack()[last_cols].resample('M').last().stack('ticker')],
                    axis=1)).dropna()

    # Calculate 5-year rolling average of dollar volume for each stocks before filtering.

    data['dollar_volume'] = (data.loc[:, 'dollar_volume'].unstack('ticker').rolling(5*12, min_periods=12).mean().stack())

    data['dollar_vol_rank'] = (data.groupby('date')['dollar_volume'].rank(ascending=False))

    data = data[data['dollar_vol_rank']<150].drop(['dollar_volume', 'dollar_vol_rank'], axis=1)

    return data



  # parlelizable
  def Calculate_Montly_Returns(self,data):

    # To capture time series dynamics that reflect, for example,
    # momentum patterns, we compute historical returns using the method
    # .pct_change(lag), that is, returns over various monthly periods as identified by lags.

    def calculate_returns(df):

      outlier_cutoff = 0.005

      lags = [1, 2, 3, 6, 9, 12]

      for lag in lags:

          df[f'return_{lag}m'] = (df['adj close']
                                .pct_change(lag)
                                .pipe(lambda x: x.clip(lower=x.quantile(outlier_cutoff),
                                                      upper=x.quantile(1-outlier_cutoff)))
                                .add(1)
                                .pow(1/lag)
                                .sub(1))
      return df


    data = data.groupby(level=1, group_keys=False).apply(calculate_returns).dropna()

    return data

  #paralizable
  def download_fama_f_and_calculate_rolling_f_b(self,data):
      factor_data = web.DataReader('F-F_Research_Data_5_Factors_2x3',
                               'famafrench',
                               start='2010')[0].drop('RF', axis=1)

      factor_data.index = factor_data.index.to_timestamp()

      factor_data = factor_data.resample('M').last().div(100)

      factor_data.index.name = 'date'

      factor_data = factor_data.join(data['return_1m']).sort_index()

      # filter out stocks whit less than 10 months of data

      observations = factor_data.groupby(level=1).size()

      valid_stocks = observations[observations >= 10]

      factor_data = factor_data[factor_data.index.get_level_values('ticker').isin(valid_stocks.index)]

      #calculate rolling factor betas

      betas = (factor_data.groupby(level=1,
                            group_keys=False)
         .apply(lambda x: RollingOLS(endog=x['return_1m'],
                                     exog=sm.add_constant(x.drop('return_1m', axis=1)),
                                     window=min(24, x.shape[0]),
                                     min_nobs=len(x.columns)+1)
         .fit(params_only=True)
         .params
         .drop('const', axis=1)))

      return betas

  def join_the_rolling_factors_data(self,betas,data):
      factors = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']

      data = (data.join(betas.groupby('ticker').shift()))

      data.loc[:, factors] = data.groupby('ticker', group_keys=False)[factors].apply(lambda x: x.fillna(x.mean()))

      data = data.drop('adj close', axis=1)

      data = data.dropna()

      return data

  def apply_pre_defined_centroids(self):

    target_rsi_values = [30, 45, 55, 70]

    initial_centroids = np.zeros((len(target_rsi_values), 18))

    initial_centroids[:, 6] = target_rsi_values

    return initial_centroids

  def k_means_clustering(self,data,initial_centroids):
    def get_clusters(df):
      df['cluster'] = KMeans(n_clusters=4,
                            random_state=0,
                            init=initial_centroids).fit(df).labels_
      return df

    data = data.dropna().groupby('date', group_keys=False).apply(get_clusters)

    return data

  def portfolio_based__on_cluster(self,data):
    filtered_df = data[data['cluster']==3].copy()

    filtered_df = filtered_df.reset_index(level=1)

    filtered_df.index = filtered_df.index+pd.DateOffset(1)

    filtered_df = filtered_df.reset_index().set_index(['date', 'ticker'])

    dates = filtered_df.index.get_level_values('date').unique().tolist()

    fixed_dates = {}

    for d in dates:

        fixed_dates[d.strftime('%Y-%m-%d')] = filtered_df.xs(d, level=0).index.tolist()

    return fixed_dates


  def optimize_weights(prices, lower_bound=0):

    returns = expected_returns.mean_historical_return(prices=prices,
                                                      frequency=252)

    cov = risk_models.sample_cov(prices=prices,
                                 frequency=252)

    ef = EfficientFrontier(expected_returns=returns,
                           cov_matrix=cov,
                           weight_bounds=(lower_bound, .1),
                           solver='SCS')

    weights = ef.max_sharpe()

    return ef.clean_weights()

  #paralelizable
  def download_fresh_daily_prices(self,data):
    stocks = data.index.get_level_values('ticker').unique().tolist()

    new_df = yf.download(tickers=stocks,
                        start=data.index.get_level_values('date').unique()[0]-pd.DateOffset(months=12),
                        end=data.index.get_level_values('date').unique()[-1],auto_adjust=False)

    return new_df
  #paralelizable
  def calculate_each_day_portfolio_return(self,new_df,fixed_dates):

    returns_dataframe = np.log(new_df['Adj Close']).diff()
    
    portfolio_df = pd.DataFrame()
    
    for start_date in fixed_dates.keys():
    
        try:
            end_date = (pd.to_datetime(start_date)+pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
            cols = fixed_dates[start_date]
            optimization_start_date = (pd.to_datetime(start_date)-pd.DateOffset(months=12)).strftime('%Y-%m-%d')
            optimization_end_date = (pd.to_datetime(start_date)-pd.DateOffset(days=1)).strftime('%Y-%m-%d')
    
            optimization_df = new_df[optimization_start_date:optimization_end_date]['Adj Close'][cols]
    
            success = False
            try:
                weights = optimize_weights(prices=optimization_df,
                                       lower_bound=round(1/(len(optimization_df.columns)*2),3))
                weights = pd.DataFrame(weights, index=optimization_df.columns, columns=[0])
                success = True
            except:
                print(f'Max Sharpe Optimization failed for {start_date}, Continuing with Equal-Weights')
    
            if success==False:
                weights = pd.DataFrame([1/len(optimization_df.columns) for i in range(len(optimization_df.columns))],
                                         index=optimization_df.columns.tolist(),
                                         columns=[0])
    
            temp_df = returns_dataframe[start_date:end_date][cols]
    
            # Create portfolio returns by calculating weighted average
            portfolio_returns = []
            for date in temp_df.index:
                daily_return = 0
                for ticker in temp_df.columns:
                    if pd.notna(temp_df.loc[date, ticker]) and ticker in weights.index:
                        daily_return += temp_df.loc[date, ticker] * weights.loc[ticker, 0]
                portfolio_returns.append(daily_return)
    
            temp_portfolio_df = pd.DataFrame(portfolio_returns,
                                           index=temp_df.index,
                                           columns=['Strategy Return'])
    
            portfolio_df = pd.concat([portfolio_df, temp_portfolio_df], axis=0)
    
        except Exception as e:
            print(f"Error for {start_date}: {e}")
    
    portfolio_df = portfolio_df.drop_duplicates()
    
    return portfolio_df


  def get_portfolio_returns_to_visualize(self,ticker,start_date,end_date,portfolio_df):

    data= yf.download(tickers=ticker,
                  start=start_date,
                  end=end_date, auto_adjust=False)
    data.columns = data.columns.droplevel(1)

    data_ret = np.log(data[['Adj Close']]).diff().dropna().rename({'Adj Close':f'{ticker} Buy&Hold'}, axis=1)

    portfolio_df = portfolio_df.merge(data_ret,
                                  left_index=True,
                                  right_index=True)
    return portfolio_df




In [5]:
#descarga sequencial

end_date = '2023-09-27'

start_date = pd.to_datetime(end_date)-pd.DateOffset(365*8)

sp500 = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0]

sp500['Symbol'] = sp500['Symbol'].str.replace('.', '-')

symbols_list = sp500['Symbol'].unique().tolist()

df_sequential= yf.download(tickers=symbols_list,
              start=start_date,
              end=end_date,auto_adjust=False)

[*********************100%***********************]  503 of 503 completed

4 Failed downloads:
['GEV', 'SW', 'SOLV', 'VLTO']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-09-29 00:00:00 -> 2023-09-27) (Yahoo error = "Data doesn\'t exist for startDate = 1443499200, endDate = 1695787200")')


In [6]:
# descarga 10 batches

end_date = '2023-09-27'

start_date = pd.to_datetime(end_date)-pd.DateOffset(365*8)

sp500 = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0]

sp500['Symbol'] = sp500['Symbol'].str.replace('.', '-')

symbols_list = sp500['Symbol'].unique().tolist()




batch_size=10

ticker_batches=np.array_split(symbols_list,batch_size)

data=[]


for tickers in ticker_batches:

  df= yf.download(tickers=tickers.tolist(),
                start=start_date,
                end=end_date,auto_adjust=False)
  data.append(df)

[*********************100%***********************]  51 of 51 completed
[*********************100%***********************]  51 of 51 completed
[*********************100%***********************]  51 of 51 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed

1 Failed download:
['GEV']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-09-29 00:00:00 -> 2023-09-27) (Yahoo error = "Data doesn\'t exist for startDate = 1443499200, endDate = 1695787200")')
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed
[*********************100%***********************]  50 of 50 completed

2 Failed downloads:
['SOLV', 'SW']: YFPricesMissingError('possibly delisted; no price data found  (1d 2015-09-29 00:00:00 -> 2023-09-27) (Yah

In [ ]:
# %%writefile profiling.py

# import cProfile
# import time
# from RollingOLSRegression import RollingOLSRegression
# import pandas as pd
# import datetime as dt


def profiling_pipeline():
  symbols_list = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'BRK-B', 'JPM', 'JNJ', 'V', 'PG',
    'MA', 'UNH', 'DIS', 'HD', 'VZ', 'NVDA', 'PYPL', 'BAC', 'ADBE', 'CMCSA',
    'NFLX', 'NKE', 'KO', 'PEP', 'MRK', 'PFE', 'XOM', 'CVX', 'WMT', 'T'
  ]

  end_date = '2023-09-27'

  start_date = pd.to_datetime(end_date)-pd.DateOffset(365*8)

  rolling_ols = RollingOLSRegression(symbols_list,start_date,end_date)

  start_time = time.time()
  df= rolling_ols.load_data()
  print(f"Data Loading Time: {time.time() - start_time} seconds")


  start_time = time.time()
  df_indicators = rolling_ols.calculate_tecnical_indicators(df)

  df_aggregate= rolling_ols.aggregate_to_monthly_level(df_indicators)

  df_montly_returns = rolling_ols.Calculate_Montly_Returns(df_aggregate)

  betas = rolling_ols.download_fama_f_and_calculate_rolling_f_b(df_montly_returns)

  join_data=rolling_ols.join_the_rolling_factors_data(betas,df_montly_returns)

  centroids=rolling_ols.apply_pre_defined_centroids()

  print(f'preprocess secuential data: {time.time()-start_time}')

  start_time=time.time()

  # data_with_clusters = rolling_ols.k_means_clustering(join_data,centroids)

  # fixed_dates = rolling_ols.portfolio_based__on_cluster(data_with_clusters)

  clusters=rolling_ols.k_means_clustering(join_data,centroids)

  fixed_dates = rolling_ols.portfolio_based__on_cluster(clusters)

  print(f'clustering time: {time.time()-start_time}')


  start_time=time.time()
  fresh_daily_prices=rolling_ols.download_fresh_daily_prices(clusters)

  portfolio_returns=rolling_ols.calculate_each_day_portfolio_return(fresh_daily_prices,fixed_dates)

  start_date='2015-01-01'

  end_date=dt.date.today()

  final_portfolio=rolling_ols.get_portfolio_returns_to_visualize('GOOGL',start_date,end_date,portfolio_returns)



  # final_portfolio
  print(f'final portfolio inference time :{time.time()-start_time}')


# if __name__ == "__main__":
#   cProfile.run("profiling_pipeline()")

profiling_pipeline()



In [48]:
# funcion auxiliar para partir en lotes

def split_df_by_tickers(df, batch_size=10):
    # Get unique tickers from level 1
    tickers = df.columns.get_level_values(1).unique()
    # print(tickers)

    
    
    # Split tickers into batches
    # ticker_batches = [tickers[i:i + batch_size] for i in range(0, len(tickers), batch_size)]
    ticker_batches=np.array_split(tickers,batch_size)
   
    
    # Create DataFrames for each batch
    df_batches = []
    for batch_tickers in ticker_batches:
        # Select columns for current batch of tickers
        batch_df = df.loc[:, (slice(None), batch_tickers)]
        df_batches.append(batch_df)
    
    return df_batches


# batches=split_df_by_tickers(df_sequential)

# batches[1].columns.get_level_values(0).unique()
# batches[0]

Price        Adj Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2015-09-29   31.251011   24.536381   35.061218         NaN  32.820747   
2015-09-30   31.797493   24.815359   36.137154         NaN  33.418999   
2015-10-01   31.251011   24.653378   36.615364         NaN  33.867683   
2015-10-02   32.112400   24.833349   37.073639         NaN  34.382847   
2015-10-05   32.732975   24.923351   37.644814         NaN  34.391163   
...                ...         ...         ...         ...        ...   
2023-09-20  111.046097  173.985001  144.022461  138.009995  96.817795   
2023-09-21  108.340797  172.438370  143.853668  132.750000  95.514412   
2023-09-22  109.999535  173.290985  143.206772  132.199997  94.761337   
2023-09-25  110.522820  174.569946  144.997559  134.139999  94.104805   
2023-09-26  109.328125  170.485275  144.303726  132.279999  92.907631   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2015-09-29  23.217773   82.010063   79.830002   44.396561  30.731047  ...   
2015-09-30  23.287504   83.419357   82.220001   46.335068  31.053207  ...   
2015-10-01  23.262148   83.572174   83.750000   45.127605  31.060698  ...   
2015-10-02  23.563267   84.378685   83.989998   45.743656  31.779905  ...   
2015-10-05  23.766125   86.271912   85.839996   47.172905  32.693890  ...   
...               ...         ...         ...         ...        ...  ...   
2023-09-20  77.327065  307.660492  535.780029  170.806656  74.535339  ...   
2023-09-21  76.347641  302.998993  513.880005  168.633743  74.065849  ...   
2023-09-22  76.984741  306.970978  512.900024  170.224625  72.281815  ...   
2023-09-25  77.165421  307.845001  511.600006  170.176117  72.582283  ...   
2023-09-26  77.270020  301.649048  506.299988  167.178650  71.427353  ...   

Price        Volume                                                         \
Ticker          ARE      ATO      AVB      AVGO      AVY      AWK     AXON   
Date                                                                         
2015-09-29   653400   743900  1233700  47991000   540500   874900  1284200   
2015-09-30   716500   788400  1099400  43955000   997400  1109500  1459200   
2015-10-01   652700   661200  1064900  45674000  1317900   925200   859500   
2015-10-02   455100   612000   598400  35207000  1034600  1192000   844700   
2015-10-05   292000   495400   640600  39085000  1133500  1180100   854900   
...             ...      ...      ...       ...      ...      ...      ...   
2023-09-20   697700   448100   500300  13681000   492200   711200   458300   
2023-09-21  2525800   615300   690000  53654000   587800  1001300   598300   
2023-09-22  2000100   731500   772500  27764000   452700  1071900   369200   
2023-09-25  1452200   772100   841400  15648000   516900   897200   373600   
2023-09-26  1352500  1189600   715300  17641000   613300  1166600   444700   

Price                                 
Ticker          AXP     AZO       BA  
Date                                  
2015-09-29  5283000  276400  2935100  
2015-09-30  4912200  403600  3062500  
2015-10-01  4741100  334300  2878400  
2015-10-02  5072400  369500  3020500  
2015-10-05  6480600  331300  4587500  
...             ...     ...      ...  
2023-09-20  1969700  161000  3738800  
2023-09-21  2712100  161700  4745700  
2023-09-22  2380600  112800  4450900  
2023-09-25  2434400  114400  3527100  
2023-09-26  2332100  128300  3666300  

[2012 rows x 306 columns]

## PARALELIZACION DE calulate_tecnical_indicators

In [50]:

@ray.remote
def calculate_tecnical_indicators(df):
    # df=copy.deepcopy(data)
    df=df.stack()
    df.index.names=['date','ticker']
    df.columns=df.columns.str.lower()


    # calculate Garman-Klass Volatility, rsi,bollinger bands, atr,macd,dollar volume
    df['garman_klass_vol'] = ((np.log(df['high'])-np.log(df['low']))**2)/2-(2*np.log(2)-1)*((np.log(df['adj close'])-np.log(df['open']))**2)

    df['rsi'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.rsi(close=x, length=20))

    df['bb_low'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,0])

    df['bb_mid'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,1])

    df['bb_high'] = df.groupby(level=1)['adj close'].transform(lambda x: pandas_ta.bbands(close=np.log1p(x), length=20).iloc[:,2])

    def compute_atr(stock_data):
        atr = pandas_ta.atr(high=stock_data['high'],
                            low=stock_data['low'],
                            close=stock_data['close'],
                            length=14)
        return atr.sub(atr.mean()).div(atr.std())

    df['atr'] = df.groupby(level=1, group_keys=False).apply(compute_atr)

    def compute_macd(close):
        macd = pandas_ta.macd(close=close, length=20).iloc[:,0]
        return macd.sub(macd.mean()).div(macd.std())

    df['macd'] = df.groupby(level=1, group_keys=False)['adj close'].apply(compute_macd)

    df['dollar_volume'] = (df['adj close']*df['volume'])/1e6


    return df











start_time=time.time();

futures=[calculate_tecnical_indicators.remote(df) for df in data]

results = ray.get(futures)

df_indicators_parallel=pd.concat(results)

end_time=time.time()

print(f'ray time: {end_time-start_time}')

df_indicators_parallel










ray time: 7.549624919891357


Price               adj close       close        high         low        open  \
date       ticker                                                               
2015-09-29 A        31.251011   33.740002   34.060001   33.240002   33.360001   
           AAPL     24.536388   27.264999   28.377501   26.965000   28.207500   
           ABBV     35.061234   52.790001   54.189999   51.880001   53.099998   
           ABT      32.820747   39.500000   40.150002   39.029999   39.259998   
           ACGL     23.217773   24.416668   24.456667   24.100000   24.170000   
...                       ...         ...         ...         ...         ...   
2023-09-26 XYL      87.701065   89.519997   90.849998   89.500000   90.379997   
           YUM     119.860703  124.010002  124.739998  123.449997  124.239998   
           ZBH     110.800163  112.459999  117.110001  112.419998  116.769997   
           ZBRA    223.960007  223.960007  226.649994  222.580002  225.970001   
           ZTS     173.604691  176.869995  178.449997  176.270004  176.580002   

Price                   volume  garman_klass_vol        rsi    bb_low  \
date       ticker                                                       
2015-09-29 A         2252400.0         -0.001351        NaN       NaN   
           AAPL    293461600.0         -0.006207        NaN       NaN   
           ABBV     12842800.0         -0.065607        NaN       NaN   
           ABT      12287500.0         -0.011997        NaN       NaN   
           ACGL      1888800.0         -0.000516        NaN       NaN   
...                        ...               ...        ...       ...   
2023-09-26 XYL       1322400.0         -0.000238  26.146716  4.474158   
           YUM       1500600.0         -0.000443  36.057158  4.792447   
           ZBH       3610500.0         -0.000229  31.893242  4.739333   
           ZBRA       355400.0          0.000133  29.494977  5.400991   
           ZTS       1463200.0         -0.000036  42.623464  5.145043   

Price                bb_mid   bb_high       atr      macd  dollar_volume  
date       ticker                                                         
2015-09-29 A            NaN       NaN       NaN       NaN      70.389777  
           AAPL         NaN       NaN       NaN       NaN    7200.487797  
           ABBV         NaN       NaN       NaN       NaN     450.284410  
           ABT          NaN       NaN       NaN       NaN     403.284933  
           ACGL         NaN       NaN       NaN       NaN      43.853730  
...                     ...       ...       ...       ...            ...  
2023-09-26 XYL     4.556072  4.637985  0.033800 -2.159189     115.975888  
           YUM     4.822408  4.852369  0.142547 -1.363696     179.862970  
           ZBH     4.778997  4.818662 -0.381708 -0.881067     400.043989  
           ZBRA    5.539167  5.677342 -0.057389 -1.600791      79.595386  
           ZTS     5.203852  5.262662  0.651515 -1.188278     254.018383  

[980418 rows x 14 columns]

## VERSION SECUENCIAL DEL calculate_tecnical_indicators

In [23]:
#hay que comentar las partes de ray en la celda anterior

start_time=time.time();

df_indicators_seq = calculate_tecnical_indicators(df_sequential)

end_time=time.time()

print(f'secuential time: {end_time-start_time}')

df_indicators_seq

#SI SIRVIO PARALELIZAR

secuential time: 17.333738565444946


Price               adj close       close        high         low        open  \
date       ticker                                                               
2015-09-29 A        31.251020   33.740002   34.060001   33.240002   33.360001   
           AAPL     24.536385   27.264999   28.377501   26.965000   28.207500   
           ABBV     35.061211   52.790001   54.189999   51.880001   53.099998   
           ABT      32.820747   39.500000   40.150002   39.029999   39.259998   
           ACGL     23.217773   24.416668   24.456667   24.100000   24.170000   
...                       ...         ...         ...         ...         ...   
2023-09-26 XYL      87.701065   89.519997   90.849998   89.500000   90.379997   
           YUM     119.860725  124.010002  124.739998  123.449997  124.239998   
           ZBH     110.800163  112.459999  117.110001  112.419998  116.769997   
           ZBRA    223.960007  223.960007  226.649994  222.580002  225.970001   
           ZTS     173.604675  176.869995  178.449997  176.270004  176.580002   

Price                   volume  garman_klass_vol        rsi    bb_low  \
date       ticker                                                       
2015-09-29 A         2252400.0         -0.001351        NaN       NaN   
           AAPL    293461600.0         -0.006207        NaN       NaN   
           ABBV     12842800.0         -0.065607        NaN       NaN   
           ABT      12287500.0         -0.011997        NaN       NaN   
           ACGL      1888800.0         -0.000516        NaN       NaN   
...                        ...               ...        ...       ...   
2023-09-26 XYL       1322400.0         -0.000238  26.146673  4.474158   
           YUM       1500600.0         -0.000443  36.057129  4.792448   
           ZBH       3610500.0         -0.000229  31.893248  4.739333   
           ZBRA       355400.0          0.000133  29.494977  5.400991   
           ZTS       1463200.0         -0.000036  42.623464  5.145043   

Price                bb_mid   bb_high       atr      macd  dollar_volume  
date       ticker                                                         
2015-09-29 A            NaN       NaN       NaN       NaN      70.389798  
           AAPL         NaN       NaN       NaN       NaN    7200.486678  
           ABBV         NaN       NaN       NaN       NaN     450.284116  
           ABT          NaN       NaN       NaN       NaN     403.284933  
           ACGL         NaN       NaN       NaN       NaN      43.853730  
...                     ...       ...       ...       ...            ...  
2023-09-26 XYL     4.556072  4.637985  0.033800 -2.159189     115.975888  
           YUM     4.822408  4.852369  0.142547 -1.363694     179.863005  
           ZBH     4.778997  4.818662 -0.381708 -0.881067     400.043989  
           ZBRA    5.539167  5.677342 -0.057389 -1.600791      79.595386  
           ZTS     5.203852  5.262662  0.651515 -1.188278     254.018361  

[980418 rows x 14 columns]

## FUNCION SECUENCIAL AGGREGATE_TO_MONTLY_LEVEL(no es conveniente paralelizar)

In [51]:
def aggregate_to_monthly_level(df):

    # To reduce training time and experiment with features and strategies,
    # we convert the business-daily data to month-end frequency.


    last_cols = [c for c in df.columns.unique(0) if c not in ['dollar_volume', 'volume', 'open',
                                                            'high', 'low', 'close']]

    data = (pd.concat([df.unstack('ticker')['dollar_volume'].resample('M').mean().stack('ticker').to_frame('dollar_volume'),
                    df.unstack()[last_cols].resample('M').last().stack('ticker')],
                    axis=1)).dropna()

    # Calculate 5-year rolling average of dollar volume for each stocks before filtering.

    data['dollar_volume'] = (data.loc[:, 'dollar_volume'].unstack('ticker').rolling(5*12, min_periods=12).mean().stack())

    data['dollar_vol_rank'] = (data.groupby('date')['dollar_volume'].rank(ascending=False))

    data = data[data['dollar_vol_rank']<150].drop(['dollar_volume', 'dollar_vol_rank'], axis=1)

    return data

df_aggregate=aggregate_to_monthly_level(df_indicators_parallel)

df_aggregate

adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2016-10-31 AAPL     26.090462         -0.002767  49.891141  3.289745   
           ABBV     38.834370         -0.056807  27.477803  3.718614   
           ABT      33.619492         -0.009785  38.008854  3.535356   
           ACN     101.760170         -0.006263  53.823690  4.619889   
           ADBE    107.510002          0.000059  53.668389  4.679513   
...                       ...               ...        ...       ...   
2023-09-30 VZ       29.285011         -0.005147  42.222505  3.404462   
           WDAY    229.240005          0.000141  43.976804  5.437299   
           WFC      38.847340         -0.000904  40.920271  3.673898   
           WMT      53.052734         -0.000186  54.722514  3.974950   
           UBER     44.270000          0.000441  45.005268  3.806654   

                     bb_mid   bb_high       atr      macd  
date       ticker                                          
2016-10-31 AAPL    3.318620  3.347495 -1.038688 -0.195978  
           ABBV    3.772733  3.826853 -0.893132 -0.760593  
           ABT     3.585802  3.636249 -1.035224 -0.650887  
           ACN     4.631524  4.643160 -0.996806 -0.135457  
           ADBE    4.694639  4.709766 -1.230331 -0.109039  
...                     ...       ...       ...       ...  
2023-09-30 VZ      3.436605  3.468747 -1.078816 -0.350386  
           WDAY    5.495242  5.553185 -0.127668 -0.306858  
           WFC     3.714239  3.754579 -0.558742 -0.282325  
           WMT     3.992416  4.009883 -0.196381  0.399459  
           UBER    3.862227  3.917801 -0.746098 -0.133973  

[12516 rows x 8 columns]

## Version Paralela calculate_montly_returns

In [60]:
# parlelizable
@ray.remote
def Calculate_Montly_Returns(data):

  # To capture time series dynamics that reflect, for example,
  # momentum patterns, we compute historical returns using the method
  # .pct_change(lag), that is, returns over various monthly periods as identified by lags.

  data=data.stack()
  data.index.names=['date','ticker']
  data.columns=data.columns.str.lower()

  def calculate_returns(df):

    outlier_cutoff = 0.005

    lags = [1, 2, 3, 6, 9, 12]

    for lag in lags:

        df[f'return_{lag}m'] = (df['adj close']
                              .pct_change(lag)
                              .pipe(lambda x: x.clip(lower=x.quantile(outlier_cutoff),
                                                    upper=x.quantile(1-outlier_cutoff)))
                              .add(1)
                              .pow(1/lag)
                              .sub(1))
    return df


  data = data.groupby(level=1, group_keys=False).apply(calculate_returns).dropna()

  return data



df= df_aggregate.unstack()

batches=split_df_by_tickers(df,10)

start_date=time.time()

futures=[Calculate_Montly_Returns.remote(df) for df in batches]

results=ray.get(futures)

montly_returns=pd.concat(results)

end_date=time.time()

print(f'calculate_Montly_returns parallel time: {end_date-start_date}')

montly_returns




calculate_Montly_returns parallel time: 0.7127950191497803


adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2017-10-31 AAPL     39.529034         -0.001263  69.196882  3.590193   
           ABBV     65.125298         -0.042703  55.247856  4.161565   
           ABT      47.540348         -0.007128  53.844930  3.873128   
           ACN     127.138969         -0.005423  69.365032  4.785195   
           ADBE    175.160004          0.000067  70.089317  4.951759   
...                       ...               ...        ...       ...   
2023-09-30 MRNA     98.120003          0.000146  38.747314  4.582514   
           NCLH     15.390000          0.000384  36.386088  2.798368   
           PLTR     13.960000          0.000214  41.544692  2.701939   
           SPGI    359.156555         -0.000037  32.594814  5.899894   
           UBER     44.270000          0.000441  45.005268  3.806654   

                     bb_mid   bb_high       atr      macd  return_1m  \
date       ticker                                                      
2017-10-31 AAPL    3.637059  3.683926 -0.906642 -0.039276   0.096807   
           ABBV    4.207901  4.254238  0.375557  0.473812   0.022728   
           ABT     3.896688  3.920248 -1.040044  0.276133   0.021276   
           ACN     4.824869  4.864543 -0.986514  0.352342   0.064180   
           ADBE    5.089292  5.226825 -0.888269  0.612102   0.174152   
...                     ...       ...       ...       ...        ...   
2023-09-30 MRNA    4.685332  4.788149 -0.529511 -0.376899  -0.132219   
           NCLH    2.865661  2.932955 -1.683386 -0.243322  -0.071213   
           PLTR    2.779743  2.857548 -0.426167 -0.433581  -0.068091   
           SPGI    5.946860  5.993827  0.220724 -1.281634  -0.068541   
           UBER    3.862227  3.917801 -0.746098 -0.133973  -0.062672   

                   return_2m  return_3m  return_6m  return_9m  return_12m  
date       ticker                                                          
2017-10-31 AAPL     0.015249   0.044955   0.028875   0.038941    0.035228  
           ABBV     0.098590   0.091379   0.056495   0.047273    0.044026  
           ABT      0.034308   0.034801   0.038672   0.031320    0.029294  
           ACN      0.048454   0.037203   0.028692   0.027398    0.018728  
           ADBE     0.062497   0.061392   0.045993   0.049515    0.041515  
...                      ...        ...        ...        ...         ...  
2023-09-30 MRNA    -0.086803  -0.068763  -0.071952  -0.064976   -0.015431  
           NCLH    -0.164939  -0.109173   0.022711   0.025772    0.025624  
           PLTR    -0.161174  -0.030723   0.087272   0.090143    0.046083  
           SPGI    -0.038235  -0.030852   0.009937   0.010146    0.015597  
           UBER    -0.053920   0.008422   0.057244   0.066838    0.043691  

[10340 rows x 14 columns]

## Version Secuencial calculate_montly_returns()

In [59]:
start_time=time.time()

result=Calculate_Montly_Returns(df_aggregate)

end_time=time.time()

print(f'calculate_Montly_returns secuencial time: {end_time-start_time}')

result



calculate_Montly_returns secuencial time: 2.1588282585144043


adj close  garman_klass_vol        rsi    bb_low  \
date       ticker                                                      
2017-10-31 AAPL     39.529034         -0.001263  69.196882  3.590193   
           ABBV     65.125298         -0.042703  55.247856  4.161565   
           ABT      47.540348         -0.007128  53.844930  3.873128   
           ACN     127.138969         -0.005423  69.365032  4.785195   
           ADBE    175.160004          0.000067  70.089317  4.951759   
...                       ...               ...        ...       ...   
2023-09-30 VZ       29.285011         -0.005147  42.222505  3.404462   
           WDAY    229.240005          0.000141  43.976804  5.437299   
           WFC      38.847340         -0.000904  40.920271  3.673898   
           WMT      53.052734         -0.000186  54.722514  3.974950   
           UBER     44.270000          0.000441  45.005268  3.806654   

                     bb_mid   bb_high       atr      macd  return_1m  \
date       ticker                                                      
2017-10-31 AAPL    3.637059  3.683926 -0.906642 -0.039276   0.096807   
           ABBV    4.207901  4.254238  0.375557  0.473812   0.022728   
           ABT     3.896688  3.920248 -1.040044  0.276133   0.021276   
           ACN     4.824869  4.864543 -0.986514  0.352342   0.064180   
           ADBE    5.089292  5.226825 -0.888269  0.612102   0.174152   
...                     ...       ...       ...       ...        ...   
2023-09-30 VZ      3.436605  3.468747 -1.078816 -0.350386  -0.056890   
           WDAY    5.495242  5.553185 -0.127668 -0.306858  -0.062413   
           WFC     3.714239  3.754579 -0.558742 -0.282325  -0.015500   
           WMT     3.992416  4.009883 -0.196381  0.399459  -0.000677   
           UBER    3.862227  3.917801 -0.746098 -0.133973  -0.062672   

                   return_2m  return_3m  return_6m  return_9m  return_12m  
date       ticker                                                          
2017-10-31 AAPL     0.015249   0.044955   0.028875   0.038941    0.035228  
           ABBV     0.098590   0.091379   0.056495   0.047273    0.044026  
           ABT      0.034308   0.034801   0.038672   0.031320    0.029294  
           ACN      0.048454   0.037203   0.028692   0.027398    0.018728  
           ADBE     0.062497   0.061392   0.045993   0.049515    0.041515  
...                      ...        ...        ...        ...         ...  
2023-09-30 VZ      -0.016122  -0.033458  -0.021495  -0.014100   -0.006158  
           WDAY    -0.016777   0.004919   0.017531   0.035597    0.034709  
           WFC     -0.057917  -0.013554   0.016712   0.000702    0.003255  
           WMT      0.010014   0.012354   0.017574   0.016553    0.020256  
           UBER    -0.053920   0.008422   0.057244   0.066838    0.043691  

[10340 rows x 14 columns]

PARALELIZACION CALCULATE_MONTLY_RETURNS

## PARALELIZACION DEL PROCESO DE ENTRENAMIENTO

In [64]:
import copy


def pre_clustering(data_original):

      data=copy.deepcopy(data_original)


      # download_fama_f_and_calculate_rolling_f_b
      factor_data = web.DataReader('F-F_Research_Data_5_Factors_2x3',
                               'famafrench',
                               start='2010')[0].drop('RF', axis=1)

      factor_data.index = factor_data.index.to_timestamp()

      factor_data = factor_data.resample('M').last().div(100)

      factor_data.index.name = 'date'

      factor_data = factor_data.join(data['return_1m']).sort_index()

      # filter out stocks whit less than 10 months of data

      observations = factor_data.groupby(level=1).size()

      valid_stocks = observations[observations >= 10]

      factor_data = factor_data[factor_data.index.get_level_values('ticker').isin(valid_stocks.index)]

      #calculate rolling factor betas

      betas = (factor_data.groupby(level=1,
                            group_keys=False)
         .apply(lambda x: RollingOLS(endog=x['return_1m'],
                                     exog=sm.add_constant(x.drop('return_1m', axis=1)),
                                     window=min(24, x.shape[0]),
                                     min_nobs=len(x.columns)+1)
         .fit(params_only=True)
         .params
         .drop('const', axis=1)))

      factors = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']

      data = (data.join(betas.groupby('ticker').shift()))

      data.loc[:, factors] = data.groupby('ticker', group_keys=False)[factors].apply(lambda x: x.fillna(x.mean()))

      data = data.drop('adj close', axis=1)

      data = data.dropna()

      return data









def apply_pre_defined_centroids_kmeans(data):

    target_rsi_values = [30, 45, 55, 70]

    initial_centroids = np.zeros((len(target_rsi_values), 18))

    initial_centroids[:, 6] = target_rsi_values


    #k_means_clustering
    def get_clusters(df):
      df['cluster'] = KMeans(n_clusters=4,
                            random_state=0,
                            init=initial_centroids).fit(df).labels_
      return df

    data = data.dropna().groupby('date', group_keys=False).apply(get_clusters)

    return data


def portfolio_based__on_cluster(data):
    filtered_df = data[data['cluster']==3].copy()

    filtered_df = filtered_df.reset_index(level=1)

    filtered_df.index = filtered_df.index+pd.DateOffset(1)

    filtered_df = filtered_df.reset_index().set_index(['date', 'ticker'])

    dates = filtered_df.index.get_level_values('date').unique().tolist()

    fixed_dates = {}

    for d in dates:

        fixed_dates[d.strftime('%Y-%m-%d')] = filtered_df.xs(d, level=0).index.tolist()

    return fixed_dates

def optimize_weights(self,prices, lower_bound=0):

    returns = expected_returns.mean_historical_return(prices=prices,
                                                      frequency=252)

    cov = risk_models.sample_cov(prices=prices,
                                 frequency=252)

    ef = EfficientFrontier(expected_returns=returns,
                           cov_matrix=cov,
                           weight_bounds=(lower_bound, .1),
                           solver='SCS')

    weights = ef.max_sharpe()

    return ef.clean_weights()

def download_fresh_daily_prices(data):
    stocks = data.index.get_level_values('ticker').unique().tolist()

    new_df = yf.download(tickers=stocks,
                        start=data.index.get_level_values('date').unique()[0]-pd.DateOffset(months=12),
                        end=data.index.get_level_values('date').unique()[-1],auto_adjust=False)

    return new_df




## Paralelizacion pre_clustering()

In [69]:
@ray.remote
def compute_beta_ray(ticker, df):
    try:
        df = df.sort_index()
        model = RollingOLS(
            endog=df['return_1m'],
            exog=sm.add_constant(df.drop('return_1m', axis=1)),
            window=min(24, df.shape[0]),
            min_nobs=len(df.columns)
        )
        result = model.fit(params_only=True)
        return (ticker, result.params.drop('const', axis=1))
    except Exception as e:
        print(f"Error in {ticker}: {e}")
        return None

def pre_clustering(data_original):
    data = copy.deepcopy(data_original)

    # Cargar factores Fama-French
    factor_data = web.DataReader('F-F_Research_Data_5_Factors_2x3',
                                 'famafrench',
                                 start='2010')[0].drop('RF', axis=1)

    factor_data.index = factor_data.index.to_timestamp()
    factor_data = factor_data.resample('M').last().div(100)
    factor_data.index.name = 'date'

    factor_data = factor_data.join(data['return_1m']).sort_index()

    # Filtrar acciones con menos de 10 observaciones
    observations = factor_data.groupby(level=1).size()
    valid_stocks = observations[observations >= 10]
    factor_data = factor_data[factor_data.index.get_level_values('ticker').isin(valid_stocks.index)]

    # Agrupar por ticker
    grouped = {ticker: df.droplevel('ticker') for ticker, df in factor_data.groupby(level=1)}

    # Ejecutar en paralelo con Ray
    futures = [compute_beta_ray.remote(ticker, df) for ticker, df in grouped.items()]
    results = ray.get(futures)

    # Filtrar nulos y reconstruir DataFrame
    results = [r for r in results if r is not None]
    betas = pd.concat([df for (_, df) in results], keys=[t for (t, _) in results], names=['ticker', 'date'])

    #join_the_rolling_factors_data


    factors = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']

    data = (data.join(betas.groupby('ticker').shift()))

    data.loc[:, factors] = data.groupby('ticker', group_keys=False)[factors].apply(lambda x: x.fillna(x.mean()))

    data = data.drop('adj close', axis=1)

    data = data.dropna()

    return data

start_time=time.time()

results= pre_clustering(montly_returns)

end_time=time.time()

print(f'pre_clustering time: {end_time-start_time}')

# clustering_data=apply_pre_defined_centroids_kmeans(results)

# fixed_dates=portfolio_based__on_cluster(clustering_data)

# new_df=download_fresh_daily_prices(clustering_data)








pre_clustering time: 1.3966803550720215


[*********************100%***********************]  155 of 155 completed


In [70]:
## version secuencial pre_clustering

def pre_clustering(data_original):

      data=copy.deepcopy(data_original)


      # download_fama_f_and_calculate_rolling_f_b
      factor_data = web.DataReader('F-F_Research_Data_5_Factors_2x3',
                               'famafrench',
                               start='2010')[0].drop('RF', axis=1)

      factor_data.index = factor_data.index.to_timestamp()

      factor_data = factor_data.resample('M').last().div(100)

      factor_data.index.name = 'date'

      factor_data = factor_data.join(data['return_1m']).sort_index()

      # filter out stocks whit less than 10 months of data

      observations = factor_data.groupby(level=1).size()

      valid_stocks = observations[observations >= 10]

      factor_data = factor_data[factor_data.index.get_level_values('ticker').isin(valid_stocks.index)]

      #calculate rolling factor betas

      betas = (factor_data.groupby(level=1,
                            group_keys=False)
         .apply(lambda x: RollingOLS(endog=x['return_1m'],
                                     exog=sm.add_constant(x.drop('return_1m', axis=1)),
                                     window=min(24, x.shape[0]),
                                     min_nobs=len(x.columns)+1)
         .fit(params_only=True)
         .params
         .drop('const', axis=1)))

      factors = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']

      data = (data.join(betas.groupby('ticker').shift()))

      data.loc[:, factors] = data.groupby('ticker', group_keys=False)[factors].apply(lambda x: x.fillna(x.mean()))

      data = data.drop('adj close', axis=1)

      data = data.dropna()

      return data

start_time=time.time();

results= pre_clustering(montly_returns)

print(f'secuential preclustering: {time.time()-start_time}')

results


#DA CASI LO MISMO


secuential preclustering: 1.6058447360992432


garman_klass_vol        rsi    bb_low    bb_mid   bb_high  \
date       ticker                                                              
2017-10-31 AAPL           -0.001263  69.196882  3.590193  3.637059  3.683926   
           ABBV           -0.042703  55.247856  4.161565  4.207901  4.254238   
           ABT            -0.007128  53.844930  3.873128  3.896688  3.920248   
           ACN            -0.005423  69.365032  4.785195  4.824869  4.864543   
           ADBE            0.000067  70.089317  4.951759  5.089292  5.226825   
...                             ...        ...       ...       ...       ...   
2023-09-30 CRWD            0.000144  51.534803  5.026187  5.103696  5.181204   
           FI              0.000065  30.683166  4.758558  4.800291  4.842024   
           FIS            -0.000278  46.954498  3.975347  4.010762  4.046176   
           MRNA            0.000146  38.747314  4.582514  4.685332  4.788149   
           UBER            0.000441  45.005268  3.806654  3.862227  3.917801   

                        atr      macd  return_1m  return_2m  return_3m  \
date       ticker                                                        
2017-10-31 AAPL   -0.906642 -0.039276   0.096807   0.015249   0.044955   
           ABBV    0.375557  0.473812   0.022728   0.098590   0.091379   
           ABT    -1.040044  0.276133   0.021276   0.034308   0.034801   
           ACN    -0.986514  0.352342   0.064180   0.048454   0.037203   
           ADBE   -0.888269  0.612102   0.174152   0.062497   0.061392   
...                     ...       ...        ...        ...        ...   
2023-09-30 CRWD   -0.744862  0.245950  -0.015641  -0.003656   0.029981   
           FI     -0.304411 -1.482810  -0.056265  -0.047270  -0.031614   
           FIS    -0.862526  0.073401   0.004790  -0.035857   0.008625   
           MRNA   -0.529511 -0.376899  -0.132219  -0.086803  -0.068763   
           UBER   -0.746098 -0.133973  -0.062672  -0.053920   0.008422   

                   return_6m  return_9m  return_12m    Mkt-RF       SMB  \
date       ticker                                                         
2017-10-31 AAPL     0.028875   0.038941    0.035228  1.277088 -0.253572   
           ABBV     0.056495   0.047273    0.044026  0.509331  0.351744   
           ABT      0.038672   0.031320    0.029294  0.836134 -0.198255   
           ACN      0.028692   0.027398    0.018728  1.204727 -0.154515   
           ADBE     0.045993   0.049515    0.041515  1.110574 -0.318697   
...                      ...        ...         ...       ...       ...   
2023-09-30 CRWD     0.026391   0.047942   -0.002216  0.590055 -0.991339   
           FI       0.002243   0.014018    0.017009  0.851777  0.462139   
           FIS      0.007031  -0.018917   -0.022479  0.968309  0.232152   
           MRNA    -0.071952  -0.064976   -0.015431  1.158717  0.838437   
           UBER     0.057244   0.066838    0.043691  1.083837  1.132621   

                        HML       RMW       CMA  
date       ticker                                
2017-10-31 AAPL   -0.613164  0.688953  0.492721  
           ABBV   -0.026958  0.164589  0.191611  
           ABT    -0.529936  0.249124  0.991747  
           ACN    -0.340296  0.265325  0.197167  
           ADBE   -0.183764 -0.218551 -0.690309  
...                     ...       ...       ...  
2023-09-30 CRWD   -0.681599 -2.499766 -0.831525  
           FI      0.202996  0.472059  0.110532  
           FIS     0.990597 -0.160500 -0.176502  
           MRNA   -1.202410  0.615315  1.063096  
           UBER   -0.239320 -1.522770 -0.481881  

[10103 rows x 18 columns]

## Paralelizacion calculate_each_day_portfolio_return

In [ ]:

@ray.remote
def calculate_return_for_date(start_date, fixed_dates, new_df, returns_dataframe):
    try:
        end_date = (pd.to_datetime(start_date)+pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
        cols = fixed_dates[start_date]
        optimization_start_date = (pd.to_datetime(start_date)-pd.DateOffset(months=12)).strftime('%Y-%m-%d')
        optimization_end_date = (pd.to_datetime(start_date)-pd.DateOffset(days=1)).strftime('%Y-%m-%d')

        optimization_df = new_df[optimization_start_date:optimization_end_date]['Adj Close'][cols]

        try:
            weights = optimize_weights(prices=optimization_df,
                                  lower_bound=round(1/(len(optimization_df.columns)*2),3))
            weights = pd.DataFrame(weights, index=optimization_df.columns, columns=[0])
        except:
            weights = pd.DataFrame([1/len(optimization_df.columns) for _ in range(len(optimization_df.columns))],
                                   index=optimization_df.columns.tolist(),
                                   columns=[0])

        temp_df = returns_dataframe[start_date:end_date][cols]
        portfolio_returns = [
            sum(temp_df.loc[date, t] * weights.loc[t, 0] for t in temp_df.columns if pd.notna(temp_df.loc[date, t]) and t in weights.index)
            for date in temp_df.index
        ]
        return pd.DataFrame(portfolio_returns, index=temp_df.index, columns=['Strategy Return'])

    except Exception as e:
        print(f"Error for {start_date}: {e}")
        return pd.DataFrame()

# NUEVA forma: paralelizar por fecha
returns_dataframe = np.log(new_df['Adj Close']).diff()

start_time=time.time();
futures = [
    calculate_return_for_date.remote(start_date, fixed_dates, new_df, returns_dataframe)
    for start_date in fixed_dates.keys()
]
results = ray.get(futures)
portfolio_df = pd.concat(results).drop_duplicates()

print(f'ray time calculate_each_day_portfolio_return: {time.time()-start_time}')

portfolio_df











ray time calculate_each_day_portfolio_return: 1.9962050914764404


,Strategy Return
Date,
2017-11-01,0.003133
2017-11-02,0.001687
2017-11-03,0.007536
2017-11-06,0.001898
2017-11-07,-0.001022
...,...
2023-09-25,0.004906
2023-09-26,-0.013776
2023-09-27,-0.003716


## Version secuencial de calculate_each_day_portfolio_return

In [ ]:
def calculate_each_day_portfolio_return(new_df,fixed_dates):

    returns_dataframe = np.log(new_df['Adj Close']).diff()

    portfolio_df = pd.DataFrame()

    for start_date in fixed_dates.keys():

        try:
            end_date = (pd.to_datetime(start_date)+pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')
            cols = fixed_dates[start_date]
            optimization_start_date = (pd.to_datetime(start_date)-pd.DateOffset(months=12)).strftime('%Y-%m-%d')
            optimization_end_date = (pd.to_datetime(start_date)-pd.DateOffset(days=1)).strftime('%Y-%m-%d')

            optimization_df = new_df[optimization_start_date:optimization_end_date]['Adj Close'][cols]

            success = False
            try:
                weights = optimize_weights(prices=optimization_df,
                                      lower_bound=round(1/(len(optimization_df.columns)*2),3))
                weights = pd.DataFrame(weights, index=optimization_df.columns, columns=[0])
                success = True
            except:
                print(f'Max Sharpe Optimization failed for {start_date}, Continuing with Equal-Weights')

            if success==False:
                weights = pd.DataFrame([1/len(optimization_df.columns) for i in range(len(optimization_df.columns))],
                                        index=optimization_df.columns.tolist(),
                                        columns=[0])

            temp_df = returns_dataframe[start_date:end_date][cols]

            # Create portfolio returns by calculating weighted average
            portfolio_returns = []
            for date in temp_df.index:
                daily_return = 0
                for ticker in temp_df.columns:
                    if pd.notna(temp_df.loc[date, ticker]) and ticker in weights.index:
                        daily_return += temp_df.loc[date, ticker] * weights.loc[ticker, 0]
                portfolio_returns.append(daily_return)

            temp_portfolio_df = pd.DataFrame(portfolio_returns,
                                          index=temp_df.index,
                                          columns=['Strategy Return'])

            portfolio_df = pd.concat([portfolio_df, temp_portfolio_df], axis=0)

        except Exception as e:
            print(f"Error for {start_date}: {e}")

    portfolio_df = portfolio_df.drop_duplicates()

    return portfolio_df


start_time=time.time();
portfolio_results=calculate_each_day_portfolio_return(new_df,fixed_dates)

print(f'secuential time calculate_each_day_portfolio_return: {time.time()-start_time}')

portfolio_results

# SI SIRVIO PARALELIZAR


Max Sharpe Optimization failed for 2017-11-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2017-12-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-01-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-02-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-03-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-04-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-05-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-06-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-07-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-08-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-09-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-10-01, Continuing with Equal-Weights
Max Sharpe Optimization failed for 2018-11-01, Continuing with Equal-Weights

,Strategy Return
Date,
2017-11-01,0.003133
2017-11-02,0.001687
2017-11-03,0.007536
2017-11-06,0.001898
2017-11-07,-0.001022
...,...
2023-09-25,0.004906
2023-09-26,-0.013776
2023-09-27,-0.003716
